<a href="https://colab.research.google.com/github/khoirulanamid/Upscale_Real_ESRGAN/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table>
  <tr>
    <td><img src="https://github.com/xinntao/Real-ESRGAN/raw/master/assets/realesrgan_logo.png" width="120"></td>
    <td><h1>4K Video Upscaler Colab (Real-ESRGAN)</h1></td>
  </tr>
</table>

[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/khoirulanamid/Upscale_Real_ESRGAN)
[![Real-ESRGAN](https://img.shields.io/badge/AI-Real--ESRGAN-orange)](https://github.com/xinntao/Real-ESRGAN)

---

### 🚀 **Selamat datang di Alat Upscale Media AI**
Notebook ini dirancang untuk mempermudah Anda meningkatkan resolusi **Video & Foto** secara otomatis hingga kualitas **4K** menggunakan algoritma **Real-ESRGAN** yang canggih.

**Author:** [khoirulanamid](https://github.com/khoirulanamid)  
**Credits:** Adapted from the original [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN) project.

---

# 1. Setup (~1 minute)

In [6]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

!pip install -q torch==2.0.1 torchvision==0.15.2 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q basicsr facexlib gfpgan ffmpeg ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

!pip install "numpy<2"
mount_drive = False

AssertionError: GPU not detected.. Please change runtime to GPU

### 🛠️ Fix Compatibility Issues
Run this cell to patch the `basicsr` library and fix the `torchvision` import error.

In [ ]:
import os

# Path to the file causing the error
file_path = '/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    # Replace the outdated import with the compatible one
    new_content = content.replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale'
    )

    with open(file_path, 'w') as f:
        f.write(new_content)
    print("✅ basicsr/data/degradations.py patched successfully!")
else:
    print("❌ Could not find the file to patch. Please ensure the setup cell was run.")

# 2. Hubungkan ke Google Drive (Opsional)
Hubungkan Colab ke Drive agar Anda bisa mengambil file video dari Drive atau menyimpan hasil upscale langsung ke sana secara permanen.

*Jika tidak diaktifkan, hasil akan tersimpan di folder sementara dan akan terhapus jika sesi Colab berakhir.*

In [ ]:
# @title Klik untuk Menghubungkan Drive
from google.colab import drive
mount_drive = True #@param{type:"boolean"}

if mount_drive:
    print("⌛ Menghubungkan ke Google Drive...")
    drive.mount('/content/gdrive/')
    print("✅ Terhubung! Anda sekarang bisa menggunakan path '/content/gdrive/MyDrive/'")
else:
    print("⚠️ Drive tidak dihubungkan. Output akan disimpan di penyimpanan sementara.")

# 3. Jalankan Upscale (Video atau Foto)
Pilih mode yang ingin Anda gunakan, masukkan path file, lalu jalankan sel ini.

In [ ]:
# @title 3. Panel Kendali Upscale (Batch Mode)
mode = "Video" #@param ["Video", "Foto"]

# @markdown ---
# @markdown ### 📁 LOKASI INPUT & OUTPUT
# @markdown *Masukkan path ke file tunggal ATAU folder untuk batch processing*
input_path = "/content/gdrive/MyDrive/Upscale_Input" #@param{type:"string"}
output_dir = "/content/gdrive/MyDrive/Upscale_Output" #@param{type:"string"}

# @markdown ---
# @markdown ### 📹 KONFIGURASI VIDEO
resolution = "4k (3840 x 2160)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"]
model_video = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]

# @markdown ---
# @markdown ### 🖼️ KONFIGURASI FOTO
model_foto = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B"]

import os, cv2

def process_video(v_path):
    if not os.path.exists(v_path): return
    video_capture = cv2.VideoCapture(v_path)
    video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_capture.release()

    if video_width == 0: return

    final_width, final_height = 1920, 1080
    if "2k" in resolution: final_width, final_height = 2560, 1440
    elif "4k" in resolution: final_width, final_height = 3840, 2160
    elif "2 x" in resolution: final_width, final_height = 2*video_width, 2*video_height
    elif "3 x" in resolution: final_width, final_height = 3*video_width, 3*video_height
    elif "4 x" in resolution: final_width, final_height = 4*video_width, 4*video_height

    scale_factor = max(final_width/video_width, final_height/video_height)
    print(f"🎬 Memproses Video: {os.path.basename(v_path)} ({video_width}x{video_height})")

    # Use absolute path to the script
    script_path = "/content/Real-ESRGAN/inference_realesrgan_video.py"
    !python {script_path} -n {model_video} -i '{v_path}' -o '{output_dir}' --outscale {scale_factor}

    video_name = os.path.splitext(os.path.basename(v_path))[0]
    potential_out = os.path.join(output_dir, f"{video_name}_out.mp4")

    if os.path.exists(potential_out):
        final_v = os.path.join(output_dir, f"{video_name}_upscaled.mp4")
        if os.path.exists(final_v): os.remove(final_v)
        os.rename(potential_out, final_v)
        print(f"🗑️ Menghapus file input: {v_path}")
        os.remove(v_path)

os.makedirs(output_dir, exist_ok=True)
files_to_process = []

if os.path.isdir(input_path):
    extensions = (".mp4", ".mkv", ".mov", ".avi") if mode == "Video" else (".jpg", ".jpeg", ".png", ".webp")
    files_to_process = [os.path.join(input_path, f) for f in os.listdir(input_path) if f.lower().endswith(extensions)]
else:
    files_to_process = [input_path]

print(f"📦 Batch Processing: Ditemukan {len(files_to_process)} file.")

for file in files_to_process:
    if mode == "Video":
        process_video(file)
    else:
        print(f"🖼️ Memproses Foto: {os.path.basename(file)}")
        script_path = "/content/Real-ESRGAN/inference_realesrgan.py"
        !python {script_path} -n {model_foto} -i '{file}' -o '{output_dir}' --outscale 4

print("✅ Semua proses batch selesai!")

# 4. Putuskan Koneksi Otomatis (Opsional)
Aktifkan ini jika Anda ingin sesi Colab otomatis berhenti setelah proses selesai. Ini sangat berguna untuk:
*   **Menghemat Kuota GPU**: Tidak membuang-buang jatah pemakaian saat Anda sedang tidak di depan komputer.
*   **Otomatisasi**: Anda bisa menjalankan proses lalu ditinggal tidur; sistem akan menutup sendiri.

*⚠️ Pastikan Anda sudah menyimpan hasil ke Google Drive (Mount Drive aktif) sebelum menggunakan fitur ini agar file tidak hilang.*

In [ ]:
from google.colab import runtime

disconnect_when_finish = True  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()